# 03 — Scorecard: hồi quy logistic trên WOE và quy đổi ra thang điểm

Ba việc: fit logistic trên các cột WOE của khối 2, kiểm hai dự đoán đã ghi trước khi mô hình hoá, rồi quy đổi hệ số ra một bảng điểm cộng được bằng tay và xuất được mã lý do.

Quy tắc chi phối cả bước: **mọi quyết định chọn biến và chọn cách chia bin đều ra đời trong `train`** bằng 5-fold CV, với WOE tính lại trong từng fold. Tập `test` không tham gia quyết định nào; nó chỉ để đối chiếu dự đoán đã ghi trước và để báo cáo Gini, calibration, ngưỡng cắt. Tập `oot` được chấm điểm cùng cả bảng ở mục 6 vì bảng điểm áp lên mọi dòng, nhưng **nhãn của nó chưa được đọc ở bất kỳ phép đo nào**; nó để dành cho khối 5.

In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd

sys.path.insert(0, str(Path.cwd().parent / 'src'))
import config, scorecard, cv_check

pd.set_option('display.width', 200); pd.set_option('display.max_columns', 50)

bins_all, woe_all, wide_sql = scorecard.load_data(drop=[])   # giu ca 10 bien de con so sanh
VARS = sorted(woe_all.variable.unique())
print(f'{len(bins_all):,} ho so, {len(VARS)} bien, {len(woe_all)} bin')
print(bins_all.groupby('split').target.agg(n='size', bad='sum', bad_rate=lambda s: round(s.mean()*100, 4)))

149,999 ho so, 10 bien, 80 bin
            n   bad  bad_rate
split                        
oot     22500  1504    6.6844
test    22500  1504    6.6844
train  104999  7018    6.6839


---
## 1. Mô hình logistic trên WOE

Hai lựa chọn cần nói rõ trước khi đọc số.

**Model log-odds của GOOD chứ không phải của BAD.** WOE ở đây định nghĩa là `ln(pct_good / pct_bad)`, nên WOE cao nghĩa là bin an toàn. Nếu model log-odds của bad thì mọi hệ số sẽ âm và không phân biệt được "âm vì quy ước" với "âm vì có vấn đề". Model good giữ được một quy tắc kiểm tra một dòng: **mọi hệ số phải dương**.

**Không regularization** (`C=1e12`). Mặc định của scikit-learn là L2 với `C=1`, đủ để kéo hệ số về 0 và làm sai lệch cả hệ số lẫn sai số chuẩn. Scorecard cần hệ số nguyên vẹn vì chúng sẽ thành điểm.

Sai số chuẩn không có sẵn trong scikit-learn nên tính tay từ Hessian: `cov = (Xᵀ W X)⁻¹` với `W = diag(p(1−p))`.

Tôi đọc ba thứ ở bảng dưới: dấu của hệ số, độ lớn so với 1, và z.

Độ lớn quanh 1 là kỳ vọng có lý do. WOE đã ở đơn vị log-odds rồi: nếu một biến là nguồn thông tin duy nhất thì hệ số của nó đúng bằng 1, vì log-odds của bin *là* dự báo. Hệ số nhỏ hơn 1 nghĩa là thông tin của biến đó đã có phần nằm trong biến khác nên model chiết khấu bớt. Hệ số lớn hơn nhiều so với 1 mới là chuyện lạ.

In [2]:
W_all = scorecard.woe_matrix(bins_all, woe_all)
tr = bins_all.split == 'train'
te = bins_all.split == 'test'
y  = 1 - bins_all.target

lr10, coef10, se10 = scorecard.fit_logit(W_all[tr], y[tr])
tab = pd.DataFrame({'he_so': coef10, 'SE': se10, 'z': coef10/se10},
                   index=['(intercept)'] + VARS).round(4)
print(tab.to_string())
for nm, m in [('train', tr), ('test', te)]:
    p = lr10.predict_proba(W_all[m])[:, 1]
    print(f'{nm:6s} Gini={scorecard.gini(y[m], p):.4f}  KS={scorecard.ks(y[m], p):.4f}')

                    he_so      SE         z
(intercept)        2.6053  0.0152  171.8246
age                0.3784  0.0319   11.8617
debt_ratio_valid   0.7891  0.0531   14.8491
dependents         0.2577  0.0758    3.4022
late_30_59         0.5201  0.0155   33.4908
late_60_89         0.3676  0.0170   21.5634
late_90            0.5158  0.0139   37.1745
monthly_income     0.0947  0.0543    1.7432
open_credit_lines -0.0462  0.0584   -0.7911
real_estate_loans  0.5578  0.0643    8.6698
revolving_util     0.6270  0.0151   41.4085


train  Gini=0.7165  KS=0.5571
test   Gini=0.6985  KS=0.5484


Trước khi fit tôi đoán Gini test rơi vào 0,60 đến 0,68, và trên 0,72 thì phải đi tìm leakage. Kết quả 0,6984 nằm ngoài khoảng đó, cao hơn cận trên. Đoán sai vì bi quan: tôi trừ hao cho phần chồng lấn giữa các biến nhiều hơn thực tế. Vẫn dưới ngưỡng báo động nên không đổi kết luận về leakage. Con số này chỉ là kỳ vọng tôi đặt ra lúc mở khối 3 chứ không nằm trong bảng giả thuyết ở `notes_credit_scoring.md` §10, nên nó không có cùng sức nặng với hai dự đoán ghi trong `results/iv_report.md`.

Chín trong mười hệ số dương, và bốn trong số đó nằm dưới 0,4, đúng như kỳ vọng chiết khấu ở trên. Chỗ phải dừng lại là **`open_credit_lines`: hệ số −0,0399, z = −0,68**.

Một hệ số âm trong quy ước này nghĩa là model đang **đảo ngược** thứ tự WOE của biến đó: những bin mà phân tích đơn biến gọi là an toàn thì trong bối cảnh đa biến lại là xấu hơn. Đây đúng là dấu hiệu tôi tự đặt ra để cảnh giác ở khối 2, và nó rơi trúng cái biến tôi bảo vệ chắc nhất, biến mà tôi đã lấy 5-fold CV để chứng minh rằng ép đơn điệu lên nó làm mất 0,0176 Gini.

Nhưng z = −0,68 nghĩa là hệ số này không phân biệt được với 0. Nói "model đảo dấu" là đọc quá tay: câu đúng là **model không tìm thấy gì ở biến này sau khi đã có chín biến kia**. Cần đo đóng góp biên thay vì đoán từ dấu.

---
## 2. Chọn biến và kiểm các dự đoán đã ghi trước

`results/iv_report.md` §2 ghi hai dự đoán trước khi dựng scorecard, cộng khoảng Gini tôi đoán ở mục trên. Kiểm bằng 5-fold CV trong train, `src/cv_check.py`.

Chỗ dễ sai nhất: **WOE phải tính lại trong từng fold**. Nếu dùng luôn cột WOE trong `features_woe`, vốn tính trên toàn bộ train, thì fold đánh giá đã góp phần tạo ra feature của chính nó. Chênh lệch đo được sẽ lạc quan, và lạc quan nhiều nhất đúng ở các phương án chia bin mảnh, tức là thiên vị đúng cái mà phép so sánh này định phân xử.

Một ngoại lệ phải nói ra: **điểm cắt bin thì không tính lại trong fold**, chúng lấy sẵn từ khối 2 (bảng `bin_cuts`, nhãn bin đọc từ `row_bins`), vốn tính trên toàn bộ train. Điểm cắt cũng là tham số học được nên lập luận ở trên áp dụng y nguyên cho chúng. Tôi để vậy vì điểm cắt là phân vị của biến, không nhìn vào nhãn, và phân vị rất ổn định khi bỏ đi 20% dữ liệu; nhưng đó là một lập luận về mức ảnh hưởng chứ không phải một lời bào chữa, và phép đo dưới đây lạc quan hơn thực tế một chút vì lý do này.

So sánh **theo cặp trên cùng fold**. Gini giữa các fold dao động từ 0,7005 đến 0,7215, biên độ 0,021, lớn gấp nhiều lần mọi chênh lệch cần đo. So hai trung bình độc lập sẽ không thấy gì; trừ theo từng fold thì phần phương sai do fold triệt tiêu.

In [3]:
base = cv_check.gini_cv(bins_all, VARS)
print(f'model 10 bien: Gini CV = {base.mean():.5f}   tung fold = {np.round(base, 4)}\n')

rows = []
for v in VARS:
    mot = cv_check.gini_cv(bins_all, [v])                          # model chi mot bien
    bo  = cv_check.gini_cv(bins_all, [x for x in VARS if x != v])  # model thieu bien do
    rows.append(dict(bien=v, don_bien=mot.mean(), khi_bo=bo.mean(),
                     **cv_check.paired(base, bo)))
loo = pd.DataFrame(rows).sort_values('d').reset_index(drop=True)
print(loo.round(5).to_string(index=False))

model 10 bien: Gini CV = 0.71513   tung fold = [0.7215 0.7211 0.7199 0.7127 0.7004]



             bien  don_bien  khi_bo        d      se         t   ktc_lo   ktc_hi  cung_dau
   revolving_util   0.56001 0.66406 -0.05107 0.00178 -28.74224 -0.05600 -0.04614      True
       late_30_59   0.38181 0.68307 -0.03206 0.00194 -16.54322 -0.03744 -0.02668      True
          late_90   0.31247 0.68827 -0.02686 0.00178 -15.08758 -0.03180 -0.02192      True
       late_60_89   0.24133 0.70283 -0.01230 0.00127  -9.71849 -0.01581 -0.00879      True
 debt_ratio_valid   0.14836 0.70882 -0.00631 0.00141  -4.48936 -0.01021 -0.00241      True
              age   0.26733 0.71152 -0.00360 0.00074  -4.88669 -0.00565 -0.00156      True
real_estate_loans   0.11936 0.71213 -0.00300 0.00090  -3.34144 -0.00549 -0.00051      True
   monthly_income   0.15349 0.71497 -0.00016 0.00005  -3.43585 -0.00029 -0.00003      True
       dependents   0.09925 0.71506 -0.00006 0.00023  -0.27604 -0.00070  0.00058     False
open_credit_lines   0.12249 0.71518  0.00006 0.00004   1.46102 -0.00005  0.00017     False

Cột `don_bien` là Gini của model chỉ có một biến đó; cột `khi_bo` là Gini của model thiếu nó; `d` là hiệu so với model đủ, nên **âm nhiều là biến đó quan trọng**. Hai cột đo trên cùng năm fold nên so được trực tiếp.

Đặt cạnh nhau thì hai cột nói hai chuyện khác nhau, và chỗ này là bài học chính của cả khối:

| biến | Gini đơn biến | đóng góp biên |
|---|---|---|
| `monthly_income` | 0,15349 | −0,00017 |
| `debt_ratio_valid` | 0,14836 | −0,00634 |
| `open_credit_lines` | 0,12249 | +0,00004 |
| `real_estate_loans` | 0,11936 | −0,00300 |
| `dependents` | 0,09925 | −0,00008 |

`monthly_income` mạnh nhất nhóm này khi đứng một mình nhưng đóng góp gần như bằng không, còn `real_estate_loans` gần yếu nhất lại đóng góp gấp gần 20 lần. Sức mạnh đơn biến đo *biến đó biết gì*; đóng góp biên đo *biến đó biết gì mà chín biến kia không biết*. Chọn biến bằng bảng IV là dùng đại lượng thứ nhất để trả lời câu hỏi thứ hai.

Tổng Gini đơn biến của mười biến là 2,41, gấp hơn ba lần Gini của model đủ. Gini không cộng được, và bảng này là cách nhìn thấy điều đó bằng số.

`open_credit_lines` có `d` **dương**: bỏ nó đi thì Gini CV nhích lên 0,00004. Quá nhỏ để gọi là cải thiện, nhưng đủ để kết luận biến này không đóng góp gì.

In [4]:
# "Ep don dieu" KHONG phai mot phuong an duy nhat. PAVA can mot chieu, va voi bien
# hinh chu U thi hai chieu cho hai ket qua khac han: ep tang giet nhanh phai, ep
# giam giet nhanh trai. De ham tu doan chieu la de mot phep do quan trong phu
# thuoc vao mot heuristic khong ai nhin. Do ca hai chieu.
U_VARS = ['revolving_util', 'open_credit_lines', 'real_estate_loans', 'debt_ratio_valid']
rows = []
for v in U_VARS:
    m = cv_check.woe_of(bins_all[bins_all.split == 'train'], v)
    cnt = bins_all[bins_all.split == 'train'].groupby(v).target.size().to_dict()
    nrm = [k for k in m.index if not cv_check.is_special(k)]
    for dr in ['tang', 'giam']:
        # so muc con lai sau khi ep: bao nhieu bin thuong bi gop phang lai
        muc = len(set(round(x, 9) for x in cv_check.force_monotone(m, cnt, dr).values()))
        g = cv_check.gini_cv(bins_all, VARS, monotone={v: dr})
        rows.append(dict(bien=v, chieu=dr, muc_con_lai=f'{muc}/{len(nrm)}',
                         gini=g.mean(), **cv_check.paired(base, g)))
print(pd.DataFrame(rows).round(5).to_string(index=False))

             bien chieu muc_con_lai    gini        d      se         t   ktc_lo   ktc_hi  cung_dau
   revolving_util  tang        1/10 0.66393 -0.05119 0.00181 -28.30730 -0.05622 -0.04617      True
   revolving_util  giam        8/10 0.71518  0.00006 0.00051   0.11050 -0.00135  0.00146     False
open_credit_lines  tang        4/10 0.71534  0.00022 0.00012   1.74115 -0.00013  0.00056     False
open_credit_lines  giam        2/10 0.71583  0.00070 0.00049   1.41953 -0.00067  0.00207     False
real_estate_loans  tang         2/4 0.71273 -0.00240 0.00061  -3.90308 -0.00410 -0.00069      True
real_estate_loans  giam         2/4 0.71516  0.00004 0.00058   0.06336 -0.00157  0.00164     False
 debt_ratio_valid  tang        1/10 0.70866 -0.00647 0.00138  -4.67652 -0.01031 -0.00263      True
 debt_ratio_valid  giam        6/10 0.71596  0.00083 0.00049   1.71392 -0.00052  0.00218     False


In [5]:
# Du doan 1, cac phep gop bin, va viec de lai tu khoi 2 (leakage cua late_90)
tests = {
    'gop bin 01 cua revolving_util':   dict(merge={'revolving_util': {'01': '02'}}),
    'bo open_credit_lines':            dict(vars_=[v for v in VARS if v != 'open_credit_lines']),
    # tach hai nhanh chu U cua real_estate_loans
    'real_estate: gop 0 vao 1':        dict(merge={'real_estate_loans': {'0': '1'}}),
    'real_estate: gop 3+ vao 2':       dict(merge={'real_estate_loans': {'3+': '2'}}),
    'real_estate: gop 0,1,2':          dict(merge={'real_estate_loans': {'0': '1', '2': '1'}}),
    'real_estate: gop het (1 bin)':    dict(merge={'real_estate_loans': {'0': '1', '2': '1', '3+': '1'}}),
    # rui ro leakage cua late_90
    'gop sentinel vao bin 0':          dict(merge={v: {'9_SENTINEL': '0'} for v in
                                                   ['late_30_59', 'late_60_89', 'late_90']}),
    'bo late_90':                      dict(vars_=[v for v in VARS if v != 'late_90']),
    'bo ca ba bien late_*':            dict(vars_=[v for v in VARS if not v.startswith('late_')]),
}
out = []
for name, kw in tests.items():
    vs = kw.pop('vars_', VARS)
    g = cv_check.gini_cv(bins_all, vs, **kw)
    out.append(dict(phuong_an=name, gini=g.mean(), **cv_check.paired(base, g)))
print(pd.DataFrame(out).round(5).to_string(index=False))

                    phuong_an    gini        d      se         t   ktc_lo   ktc_hi  cung_dau
gop bin 01 cua revolving_util 0.71525  0.00012 0.00048   0.24981 -0.00120  0.00144     False
         bo open_credit_lines 0.71518  0.00006 0.00004   1.46102 -0.00005  0.00017     False
     real_estate: gop 0 vao 1 0.71340 -0.00173 0.00062  -2.78867 -0.00346 -0.00001     False
    real_estate: gop 3+ vao 2 0.71355 -0.00158 0.00045  -3.50125 -0.00284 -0.00033      True
       real_estate: gop 0,1,2 0.71516  0.00004 0.00058   0.06336 -0.00157  0.00164     False
 real_estate: gop het (1 bin) 0.71213 -0.00300 0.00090  -3.34144 -0.00549 -0.00051      True
       gop sentinel vao bin 0 0.71277 -0.00235 0.00065  -3.63458 -0.00415 -0.00056      True
                   bo late_90 0.68827 -0.02686 0.00178 -15.08758 -0.03180 -0.02192      True
         bo ca ba bien late_* 0.59048 -0.12465 0.00217 -57.49855 -0.13067 -0.11863      True


In [6]:
# Gop bin 01 co lam revolving_util don dieu khong: xem lai bang WOE sau khi gop.
# WOE o day lech chut so voi woe_lookup vi cv_check.woe_of co Laplace +0,5 con SQL
# thi khong. Voi bin muoi nghin dong thi lech o hang thu tu, khong doi ket luan.
g = bins_all[bins_all.split == 'train'].copy()
g['revolving_util'] = g.revolving_util.replace({'01': '02'})
w = cv_check.woe_of(g, 'revolving_util').sort_index()
n = g.groupby('revolving_util').target.agg(n='size', bad='sum')
for k in w.index:
    print(f'  {k:14s} n={n.loc[k,"n"]:6d}  bad={n.loc[k,"bad"]/n.loc[k,"n"]*100:6.2f}%  WOE={w[k]:+.4f}')

  02             n= 20966  bad=  1.91%  WOE=+1.3031
  03             n= 10482  bad=  1.38%  WOE=+1.6277
  04             n= 10482  bad=  1.83%  WOE=+1.3432
  05             n= 10482  bad=  2.42%  WOE=+1.0580
  06             n= 10482  bad=  3.42%  WOE=+0.7052
  07             n= 10482  bad=  5.27%  WOE=+0.2533
  08             n= 10482  bad=  8.64%  WOE=-0.2782
  09             n= 10482  bad= 16.47%  WOE=-1.0119
  10             n= 10482  bad= 23.57%  WOE=-1.4596
  X_IMPLAUSIBLE  n=   177  bad=  7.91%  WOE=-0.2130


Mục này tôi viết lại hai lần, vì hai lỗi khác nhau trong cùng một hàm mười dòng của chính tôi. Cả hai ghi ở cuối mục.

Điều phải nói trước, vì nó là chỗ tôi đã trượt: **"ép đơn điệu" không phải một phương án duy nhất.** PAVA cần một chiều, và với biến hình chữ U thì hai chiều gộp phẳng hai nhánh khác nhau. Bảng trên đo cả hai chiều, kèm cột `muc_con_lai` cho biết sau khi ép thì còn lại mấy mức WOE phân biệt trên tổng số bin thường. Cột đó quan trọng ngang cột `d`, vì nó cho biết cái giá phải trả về mặt cấu trúc.

### Với hai biến nhiều bin, ép sai chiều không phải là ép mà là xoá biến

Nhìn cột `muc_con_lai` ở chiều tăng: `revolving_util` và `debt_ratio_valid` đều còn **1 mức trên 10**, tức PAVA gộp toàn bộ bin thường thành một hằng số. Đó không còn là một biến nữa. Và bảng đóng góp biên xác nhận đúng như vậy:

| biến | ép sai chiều (tăng) | bỏ hẳn biến |
|---|---|---|
| `revolving_util` | −0,05133 | −0,05109 |
| `debt_ratio_valid` | −0,00628 | −0,00634 |

Trùng nhau tới chữ số thứ tư, qua hai đường đi hoàn toàn khác nhau: một bên là PAVA gộp bin rồi fit lại, một bên là bỏ cột khỏi ma trận. Đây là phép kiểm chéo tình cờ có được cho cả bộ máy CV, và tôi ghi lại vì cả khối này xoay quanh chuyện phép kiểm phải có đường để fail.

### Ép đúng chiều thì mất bao nhiêu

Đọc cột `d` ở chiều giảm, kèm khoảng tin cậy 95% tính từ SE của năm fold (`d ± 2,776·se`, `t` bảng với 4 bậc tự do):

| biến | mức còn lại | d | KTC 95% của d | cái giá xấu nhất |
|---|---|---|---|---|
| `revolving_util` | 8/10 | +0,00006 | [−0,00133; +0,00146] | 0,00133 |
| `debt_ratio_valid` | 6/10 | +0,00085 | [−0,00052; +0,00222] | 0,00052 |
| `real_estate_loans` | 2/4 | +0,00006 | [−0,00154; +0,00166] | 0,00154 |
| `open_credit_lines` | 2/10 | −0,00006 | [−0,00042; +0,00030] | 0,00042 |

Cột cuối rất dễ lấy nhầm đầu, và tôi đã lấy nhầm một lần. `d = Gini(phương án) − Gini(model đủ)`, nên **cái giá là −d**, và cái giá xấu nhất mà dữ liệu còn cho phép là **−ktc_lo**, tức đầu *trái* của khoảng. Đầu phải là mức được lợi tối đa: với `debt_ratio_valid` đầu phải là 0,0022 trong khi cái giá xấu nhất thật ra chỉ 0,0005, nên lấy nhầm đầu vừa sai chiều vừa tự làm yếu kết luận của mình.

Câu đúng **không** phải "tốn 0,000 Gini", vì dữ liệu không nói được điều đó. Câu đúng là: cái giá xấu nhất còn tương thích với dữ liệu ở mức tin cậy 95% là **0,0015 Gini** trên cả bốn biến, và **0,0013** nếu chỉ tính hai biến đáng đem đi dùng. Con số đó nhỏ hơn một bậc so với 0,0063 đến 0,0205 mà khối 2 đo đơn biến, và nhỏ hơn nhiều so với ±0,028 là khoảng tin cậy của chính Gini trên tập OOT. Ở quy mô dữ liệu này nó không đo được.

Cột mức còn lại tách bảng này thành hai nhóm rất khác nhau, và đây mới là chỗ có nội dung:

**`revolving_util` và `debt_ratio_valid` giữ được 8 và 6 mức**, tức gần hết cấu trúc, mà vẫn đơn điệu và vẫn không mất gì đo được. Đây là kết quả đáng đem đi dùng: hai biến này có bảng bin đơn điệu nhiều mức, sẵn sàng cho một scorecard giải thích được.

**`real_estate_loans` chỉ đơn điệu được bằng cách rút về nhị phân** ({0, 1, 2} so với {3+}). Không mất gì, nhưng "đơn điệu 2 mức" không cùng một loại kết quả với "đơn điệu 8 mức", và nói gọn cả bốn biến vào một câu là làm loãng nó.

**`open_credit_lines` thì câu hỏi gần như rỗng.** Bản đơn điệu của nó gộp bin 01 đến 09 thành một mức, còn đúng 2 mức trên 10, và `d` = −0,00006 gần trùng với `d` = +0,00004 của việc **xoá hẳn biến**. Nói "tồn tại cách chia bin đơn điệu không mất gì" cho biến này chỉ là nói lại rằng biến này không mang gì. Nó cũng là biến duy nhất mà chiều giảm **không** phải chiều nghiệp vụ: bin 01 có bad rate 10,77%, cao nhất của biến, nên "rủi ro tăng theo số hạn mức" là sai chiều với dữ liệu, và đúng vì thế mà bản đơn điệu của nó phải gộp phẳng chín bin đầu. Biến này bị loại khỏi model cuối.

### Phán quyết

**Dự đoán 2 sai.** Tôi đoán chữ U của `open_credit_lines` và `real_estate_loans` sống sót trong model đa biến. Với `open_credit_lines`, ép đơn điệu chiều nào cũng không mất gì, và bản thân biến cũng không mang gì. Với `real_estate_loans`, chữ U rút được về nhị phân mà không mất gì, nên hình chữ U không phải thứ mang thông tin; cái mang thông tin là contrast "có từ ba khoản bất động sản trở lên hay không".

Ở khối 2, `open_credit_lines` được đo là đáng **0,0176 Gini** khi so đơn biến, ép chiều tăng. `real_estate_loans` thì bảng CV của khối 2 **không có dòng nào**, tôi chỉ đọc hình chữ U của nó từ bảng bad rate, và đó chính là cách đọc mà khối này bác.

Vì sao `open_credit_lines` không mang gì: nhánh trái của nó là nhóm bị hạn chế tín dụng, utilization trung vị 0,439 so với 0,137 ở đáy chữ U, mà `revolving_util` là biến mạnh nhất bảng. Model không cần biết một người chỉ có 2 hạn mức, nó đã biết người đó dùng gần hết hạn mức đang có.

**Dự đoán 1 đúng một nửa, và nửa sai có một lý do rất cụ thể.** Gộp bin 01 của `revolving_util` vào bin 02 không làm mất gì (`d` = +0,00011), đúng như tôi đoán. Nhưng nó không làm biến đơn điệu: nhóm gộp có bad rate 1,91%, vẫn cao hơn bin 03 (1,38%).

PAVA chiều giảm thì làm được, và nó cũng gộp bin, chỉ là **gộp 01+02+03** thành một mức có bad rate 1,73%, thấp hơn bin 04 (1,83%) nên đơn điệu. Nói cách khác tôi đã gộp **thiếu đúng một bin**. Đó là phán quyết chính xác cho dự đoán 1, và ai cũng kiểm được nó ngay trên bảng bad rate: 1,91% so với 1,38% thì hỏng, 1,73% so với 1,83% thì xong.

### Hệ quả: scorecard này có thể làm đơn điệu mà không mất gì đo được

Đây là kết quả có giá trị thực tế nhất của mục này, và nó ngược với kết luận tôi ghi ở khối 2. Đơn điệu là chuẩn mực của scorecard chính vì nó làm mã lý do nói được thành câu: "điểm của anh thấp vì tỉ lệ sử dụng hạn mức cao", không kèm ngoại lệ "trừ khi anh dùng quá ít". Ở khối 2 tôi từ chối đơn điệu vì đo được nó tốn 0,0063 đến 0,0205 Gini; đo lại trong model đa biến thì cái giá đó chìm dưới ngưỡng đo được.

Cần nói rõ nó sửa được cái gì và không sửa được cái gì. Nó xoá cái móc ở bin 01 của `revolving_util`, tức xoá đúng chỗ tôi không giải thích được với khách hàng. Nó **không** sửa được việc 84,7% hồ sơ bị từ chối nhận cùng một lý do ở mục 8: nguyên nhân của con số đó là `revolving_util` chi phối model, và ép đơn điệu chỉ rút biên độ của biến đó xuống chút ít chứ không đổi bản chất. Hai vấn đề khác nhau.

Tôi **không** đổi cách chia bin trong khối này, vì bảng WOE và toàn bộ pipeline SQL của khối 2 đang là nền cho mọi con số phía dưới, và đổi nền ở cuối khối là cách chắc chắn nhất để một chỗ nào đó lệch mà không ai biết. Đây là việc đầu tiên của khối 4.

### Hai lỗi tôi đã mắc ở chính phép kiểm này

**Lỗi thứ nhất.** Bản đầu của `_pava` chọn bin để ép bằng `str(k).isdigit()`. `'3+'.isdigit()` là `False`, nên với `real_estate_loans` (bin `0, 1, 2, 3+`) hàm chỉ ép trên `0, 1, 2` và bỏ nguyên bin `3+` ra ngoài. Kết quả in ra là `d` = −0,00015, một con số nhỏ và hợp lý.

**Lỗi thứ hai.** Sau khi sửa, hàm tự chọn chiều bằng dấu hiệp phương sai có trọng số. Với `real_estate_loans` nó chọn tăng, ra `d` = −0,00241, và tôi kết luận "chữ U sống sót". Chiều là một lựa chọn có hậu quả lớn hơn cả bản thân phép ép, mà tôi để nó cho một heuristic không in ra đâu cả. `force_monotone` bây giờ **bắt buộc** truyền chiều, không còn chế độ tự đoán.

Cùng họ với lỗi IV sai 10 lần ở khối 2: phép kiểm chạy trót lọt, in ra con số trông hợp lý, không có gì báo rằng nó đang đo một thứ khác với thứ tôi nghĩ. Cả ba lần đều lộ ra nhờ đối chiếu chứ không nhờ chạy lại.

### Hai điều phải nói kèm về mặt thống kê

**Đa so sánh.** Khối này chạy 27 phép kiểm ghép cặp trong ba bảng (10 dòng đóng góp biên, 8 dòng hai chiều, 9 dòng gộp bin), và với mỗi biến tôi báo cả hai chiều rồi bàn về chiều rẻ hơn. Nếu đó là dò số thì các t quanh 2 đến 4 mất giá trị (Benjamini & Hochberg 1995; Berk và cộng sự 2013). Nhưng chiều ở đây **không phải kết quả của việc dò**: chiều "rủi ro tăng dần" là chiều duy nhất triển khai được trong một scorecard, nên đó là chiều cần đo từ đầu, còn chiều kia chỉ để định giá việc chọn sai. Với `open_credit_lines` tôi nói thẳng ở trên rằng chiều đó không đúng nghiệp vụ, thay vì lấy con số rẻ hơn rồi im.

**t-test trên k-fold đánh giá thấp phương sai**, vì năm tập fit chồng nhau tới 75% nên các `d` không độc lập (Dietterich 1998; Bengio & Grandvalet 2004). Hệ quả là |t| bị thổi lên. Chiều lệch này **có lợi** cho các kết luận null của tôi: một phép kiểm vốn dễ bác mà vẫn không bác được thì câu "không đo được" càng chắc. Ngược lại nó làm yếu đúng những khẳng định dương nhỏ, cụ thể là `monthly_income` (t = −3,39) và "gộp 3+ vào 2" (t = −3,57). Tôi không lấy hai con số đó làm căn cứ quyết định. Chỗ tôi **có** dựa vào một t cùng cỡ là dòng "gộp hết" của `real_estate_loans` (t = −3,37) để nói biến này mang thông tin; ở đó tôi có một bằng chứng độc lập không dính CV, là hệ số Wald của biến trong model cuối, z = 8,66 trên toàn bộ train.

### Quyết định

**`open_credit_lines` bị loại**, vì đóng góp biên bằng không (+0,00004) đo bằng CV trong train. Hệ số âm ở mục 1 **không phải lý do thứ hai độc lập**: z = −0,68 nên nó không phân biệt được với 0, và nó là cùng một sự việc nhìn từ góc khác. Nó là hệ quả không bảo vệ được về mặt nghiệp vụ: giữ biến lại thì bảng điểm sẽ trừ điểm người có ít hạn mức hơn, và tôi không có câu trả lời nào đúng cho câu hỏi đó.

`monthly_income` (−0,00017) và `dependents` (−0,00008) cũng gần bằng không nhưng **giữ lại**: dấu đúng, đóng góp không âm, và là hai biến bên kinh doanh mong thấy trong một scorecard. Bỏ chúng đổi lấy 0,0002 Gini là đổi một câu hỏi khó lấy một con số không đo được.

Khối 2 còn để lại một cảnh báo: chỗ chồng lấn thật nằm ở cặp `debt_ratio_valid` và `monthly_income` (31.365 dòng dùng chung thông tin missing), và nếu có hệ số lạ thì nhìn ở đó trước. Nửa đúng: `monthly_income` đúng là bị chiết khấu gần hết (hệ số 0,0884, z = 1,64) trong khi `debt_ratio_valid` giữ nguyên sức mạnh (0,8030); nhưng chỗ **lật dấu** lại rơi vào một biến không nằm trong cảnh báo đó.

### Đọc bảng gộp bin của `real_estate_loans`

Bốn dòng `real_estate` trong bảng thứ hai phải đọc cùng nhau, vì đọc lẻ thì mâu thuẫn:

| phép gộp | bảng bin còn lại | d |
|---|---|---|
| gộp 0 vào 1 | {0,1} −0,024 · {2} +0,188 · {3+} −0,256 | −0,00174 |
| gộp 3+ vào 2 | {0} −0,238 · {1} +0,260 · {2,3+} +0,065 | −0,00156 |
| gộp 0,1,2 | nhóm 0+1+2 +0,021 · bin 3+ −0,256 | +0,00007 |
| gộp hết | một bin | −0,00300 |

Dòng thứ ba nói toàn bộ đóng góp của biến nằm ở contrast `{3+}` so với phần còn lại. Nhưng dòng thứ hai xoá đúng contrast đó mà chỉ mất một nửa, và lý do là nó **không xoá hẳn**: bin gộp `{2, 3+}` có WOE +0,065, vẫn nằm dưới bin 1 (+0,260), nên một phiên bản pha loãng của cùng contrast sống sót.

Dòng đầu mất 0,00174 dù nhánh trái được cho là không mang gì. Cơ chế: gộp bin 0 (WOE −0,238) với bin 1 (+0,260) tạo ra một contrast giả giữa nhóm gộp (−0,024) và bin 2 (+0,188), và một hệ số duy nhất buộc phải quy contrast giả đó thành điểm. Hai dòng đầu vì vậy không đo "giá trị của một nhánh", chúng đo hậu quả của việc gộp hai bin ngược dấu; chỉ dòng ba và dòng bốn mới trả lời được câu hỏi ban đầu.

Dòng "gộp 0,1,2" cho `d` = +0,00007 còn "ép giảm" ở bảng trên cho +0,00006, và hai con số trùng nhau **không phải tình cờ**: cả hai đều rút biến xuống đúng hai mức, mà một cột chỉ có hai mức thì mọi cách gán giá trị đều sai khác nhau một phép biến đổi affine, nên logistic fit ra cùng một model. Chênh 1e-05 còn lại là do hai đường tính WOE khác nhau, một bên tính lại trên bảng đã gộp (+0,021) và một bên lấy trung bình có trọng số của khối PAVA (+0,044).

---
## 3. Model cuối

Chín biến, chia bin giữ nguyên như khối 2, không gộp bin nào.

In [7]:
bins, woe, _ = scorecard.load_data()          # drop mac dinh = config.SCORECARD_DROP
COLS = sorted(woe.variable.unique())
W = scorecard.woe_matrix(bins, woe)

# doi chieu: ma tran dung lai tu woe_lookup phai trung khop features_woe cua khoi 2
# reindex theo id chu khong so theo vi tri dong, va kiem NaN truoc khi lay max:
# neu so theo vi tri thi phep doi chieu nay khong the fail dung o truong hop no
# sinh ra de bat, con neu de NaN loi vao thi max() co the van tra ve 0
ref = wide_sql.set_index('id')
lech = 0.0
for v in COLS:
    col = ref['woe_' + ('debt_ratio' if v == 'debt_ratio_valid' else v)].reindex(W.index)
    assert not col.isna().any(), f'{v}: co id trong row_bins ma khong co trong features_woe'
    lech = max(lech, float(np.abs(W[v].values - col.values).max()))
print(f'doi chieu voi features_woe: lech lon nhat = {lech:.2e}\n')

lr, coef, se = scorecard.fit_logit(W[tr], y[tr])
print(pd.DataFrame({'he_so': coef, 'SE': se, 'z': coef/se},
                   index=['(intercept)'] + COLS).round(4).to_string())
for nm, m in [('train', tr), ('test', te)]:
    p = lr.predict_proba(W[m])[:, 1]
    print(f'{nm:6s} n={int(m.sum()):6d}  Gini={scorecard.gini(y[m], p):.4f}  KS={scorecard.ks(y[m], p):.4f}')

doi chieu voi features_woe: lech lon nhat = 0.00e+00

                    he_so      SE         z
(intercept)        2.6052  0.0152  171.8416
age                0.3777  0.0319   11.8466
debt_ratio_valid   0.7969  0.0522   15.2637
dependents         0.2626  0.0755    3.4791
late_30_59         0.5219  0.0153   34.0035
late_60_89         0.3680  0.0170   21.6010
late_90            0.5143  0.0137   37.4431
monthly_income     0.0901  0.0540    1.6683
real_estate_loans  0.5477  0.0631    8.6822
revolving_util     0.6239  0.0146   42.6105
train  n=104999  Gini=0.7165  KS=0.5576
test   n= 22500  Gini=0.6984  KS=0.5474


Mọi hệ số dương. Gini test 0,6984, bằng đúng model 10 biến. Đây là báo cáo chứ không phải căn cứ: quyết định bỏ `open_credit_lines` đã ra đời ở mục 2 bằng CV trong train, và nếu test có nói ngược thì tôi vẫn phải giữ quyết định đó rồi ghi lại mâu thuẫn, chứ không được đổi ý theo test.

`monthly_income` có z = 1,64, tức p ≈ 0,10, không có ý nghĩa thống kê ở mức 5%. Chỗ này có một mâu thuẫn bề ngoài đáng nói: CV bảo bỏ nó làm Gini giảm 0,00017 với t = −3,39, tức là "có ý nghĩa", trong khi kiểm định Wald bảo hệ số không phân biệt được với 0. Hai phép kiểm này hỏi hai câu khác nhau. Wald hỏi hệ số có khác 0 không trên một mẫu train; CV ghép cặp hỏi việc bỏ biến có làm giảm Gini nhất quán qua các fold không, và vì ghép cặp nên nó phát hiện được cả những chênh lệch cực nhỏ. Cả hai cùng đúng, và cùng nói một điều: **biến này gần như không có tác dụng**. Con số 0,00017 Gini không phải là lý do giữ nó; lý do giữ nằm ở đoạn trên.

---
## 4. Quy đổi ra thang điểm

Model đã xong, nhưng `log-odds = 2,607 + 0,625·WOE + ...` không phải thứ đưa cho nhân viên tín dụng được. Cần một thang điểm mà người ta cộng được bằng tay và đọc được ý nghĩa.

Thang điểm scorecard quy ước điểm là hàm **tuyến tính của log-odds**:

$$\text{score} = \text{Offset} + \text{Factor} \cdot \ln(\text{odds})$$

với `odds` là tỉ lệ good trên bad. Tuyến tính theo log-odds chứ không theo xác suất, vì như vậy "thêm bao nhiêu điểm" mới có nghĩa cố định trên toàn thang: cộng cùng một số điểm luôn nhân odds với cùng một hệ số, dù đang ở đầu nào của thang.

Ba tham số quy ước, đặt tên theo Siddiqi (2017):

- **PDO** (points to double the odds): bao nhiêu điểm thì odds gấp đôi. Thông lệ là 20.
- **Base odds**: odds tại điểm quy chiếu. Thông lệ 50:1, tức 50 người tốt trên 1 người xấu.
- **Base score**: điểm quy chiếu đó. Thông lệ 600.

Giải ra Factor và Offset từ hai ràng buộc:

$$600 = \text{Offset} + \text{Factor}\ln 50 \qquad 620 = \text{Offset} + \text{Factor}\ln 100$$

Trừ hai vế: `20 = Factor·(ln 100 − ln 50) = Factor·ln 2`, nên `Factor = PDO/ln 2` và `Offset = 600 − Factor·ln 50`.

**Ba hằng số này không đổi thứ tự xếp hạng.** Điểm là biến đổi tuyến tính đơn điệu của log-odds nên Gini, KS, AUC đều không đổi dù chọn bộ nào. Sau bước làm tròn về số nguyên thì PDO có ảnh hưởng đến độ chính xác của PD đọc từ điểm, xem mục 6. Chúng chỉ quyết định con số hiện ra trông như thế nào. Chọn theo thông lệ để người đọc quen mắt, chứ 600 và 50:1 không có tính chất toán học nào cả.

Bước còn lại là **chia tổng điểm về từng bin**. Viết lại:

$$\text{score} = \text{Offset} + \text{Factor}\Big(\beta_0 + \sum_{j=1}^{n} \beta_j \text{WOE}_j\Big) = \sum_{j=1}^{n} \Big[\big(\beta_j \text{WOE}_j + \tfrac{\beta_0}{n}\big)\text{Factor} + \tfrac{\text{Offset}}{n}\Big]$$

Mỗi ngoặc vuông là điểm của một biến. Việc chia `β₀` và `Offset` đều cho n biến là **quy ước cho tiện**, không phải kết quả toán học: nó chỉ dời điểm qua lại giữa các biến chứ không đổi tổng. Hệ quả khi đọc bảng điểm: **so sánh điểm tuyệt đối giữa hai biến khác nhau là vô nghĩa**, chỉ *chênh lệch trong cùng một biến* mới có nghĩa.

In [8]:
factor, offset = scorecard.scaling_constants()
print(f'Factor = {config.PDO}/ln2 = {factor:.4f}')
print(f'Offset = {config.SCORE_BASE} - Factor*ln({config.ODDS_BASE}) = {offset:.4f}')

points = scorecard.build_points(coef, COLS, woe, factor, offset)
rng = points.groupby('variable').diem.agg(['min', 'max'])
rng['bien_do'] = rng['max'] - rng['min']
print(f'\nbang diem: {len(points)} dong, tong diem {rng["min"].sum()} den {rng["max"].sum()}\n')
print(rng.sort_values('bien_do', ascending=False).to_string())

Factor = 20/ln2 = 28.8539
Offset = 600 - Factor*ln(50) = 487.1229

bang diem: 70 dong, tong diem 385 den 640

                   min  max  bien_do
variable                            
revolving_util      36   93       57
late_90             16   68       52
late_30_59          19   71       52
late_60_89          30   66       36
debt_ratio_valid    49   70       21
age                 56   75       19
real_estate_loans   58   67        9
dependents          60   66        6
monthly_income      61   64        3


In [9]:
for v in COLS:
    t = points[points.variable == v].sort_values('bin')
    hi = t.diem.max()
    print(f'\n{v}   (bien do {t.diem.min()}-{hi} diem)')
    for r in t.itertuples():
        print(f'   {r.bin:14s} n={r.n:6d}  bad={r.n_bad/r.n*100:6.2f}%  WOE={r.woe:+7.4f}  '
              f'diem={r.diem:3d}  thieu {hi-r.diem:3d}')


age   (bien do 56-75 diem)
   01             n= 12007  bad= 11.52%  WOE=-0.5974  diem= 56  thieu  19
   02             n= 10401  bad=  9.53%  WOE=-0.3855  diem= 58  thieu  17
   03             n= 11139  bad=  8.47%  WOE=-0.2556  diem= 60  thieu  15
   04             n= 10319  bad=  8.15%  WOE=-0.2142  diem= 60  thieu  15
   05             n= 10402  bad=  7.84%  WOE=-0.1713  diem= 61  thieu  14
   06             n=  9918  bad=  6.31%  WOE=+0.0613  diem= 63  thieu  12
   07             n=  9352  bad=  4.88%  WOE=+0.3346  diem= 66  thieu   9
   08             n= 11443  bad=  4.10%  WOE=+0.5164  diem= 68  thieu   7
   09             n= 10018  bad=  2.76%  WOE=+0.9275  diem= 73  thieu   2
   10             n= 10000  bad=  2.18%  WOE=+1.1675  diem= 75  thieu   0

debt_ratio_valid   (bien do 49-70 diem)
   01             n=  8303  bad=  4.91%  WOE=+0.3264  diem= 70  thieu   0
   02             n=  8303  bad=  6.95%  WOE=-0.0418  diem= 62  thieu   8
   03             n=  8302  bad=  6.49%  WO

Biên độ điểm của một biến chính là **tầm quan trọng của nó trong quyết định**, và đây là dạng đọc được nhất của hệ số: `revolving_util` chênh 57 điểm giữa bin tốt nhất và xấu nhất, tức 2,85 lần PDO, nên chỉ riêng biến này đã làm odds chênh 2^2,85 ≈ 7 lần. `monthly_income` chênh 3 điểm, gần như không tham gia quyết định.

Tổng điểm chạy từ 385 đến 640. Thang hẹp một cách bất thường so với các thang điểm thương mại (FICO 300 đến 850) vì nó là *hệ quả* của model chứ không phải thiết kế: biên độ bằng `Factor` nhân biên độ log-odds mà chín biến này tạo ra được.

Hai chi tiết trong bảng cần chỉ ra vì chúng sẽ quay lại ở phần mã lý do.

`revolving_util` bin 01 được 81 điểm, **thiếu 12 điểm** so với bin 02. Đó là cái móc ở khối 2, và giờ nó có giá cụ thể: 12 điểm là 0,6 PDO, tức odds thấp hơn khoảng 1,5 lần. Người có ít hạn mức nhất *bị trừ điểm*, và bảng điểm nói thẳng điều đó ra thay vì giấu trong hệ số.

`monthly_income` bin `X_ZERO` được **điểm cao nhất** của biến, cao hơn cả nhóm thu nhập cao nhất. Đây là hệ quả trực tiếp của phát hiện ở khối 1 rằng missing mang thông tin ngược trực giác. Nó đúng về mặt thống kê trên bộ này và sẽ là câu hỏi khó đầu tiên bất kỳ ai nhìn bảng điểm cũng hỏi. Vì biên độ cả biến chỉ 3 điểm nên nó không đổi được quyết định nào, nhưng vẫn phải giải thích được.

### Kiểm điểm có quy ngược về đúng model không

Bảng điểm chỉ dùng được nếu cộng tay ra đúng cái model nói. Hai phép kiểm: điểm chưa làm tròn phải quy ngược ra đúng log-odds, và làm tròn về số nguyên phải không làm sai PD quá mức chấp nhận được.

In [10]:
lo_model = coef[0] + W.values @ coef[1:]
lo_score = (scorecard.apply_points(bins, points, col='diem_raw').values - offset) / factor
print(f'diem chua lam tron -> log-odds : lech lon nhat = {np.abs(lo_model - lo_score).max():.2e}')

score   = scorecard.apply_points(bins, points)
pd_model = 1 / (1 + np.exp(lo_model))
pd_round = scorecard.score_to_pd(score.values, factor, offset)
sai_so = np.abs(pd_model - pd_round)
j = int(sai_so.argmax())
print(f'lam tron ve so nguyen -> PD    : lech lon nhat = {sai_so.max():.5f}, '
      f'trung binh = {sai_so.mean():.6f}')
print(f'   cho lech lon nhat: {score.values[j]:.0f} diem, PD model {pd_model[j]*100:.1f}% '
      f'so voi {pd_round[j]*100:.1f}% doc tu diem')
for lo_, hi_ in [(0, 480), (480, 520), (520, 600), (600, 700)]:
    m_ = (score.values >= lo_) & (score.values < hi_)
    print(f'   diem {lo_}-{hi_}: n={m_.sum():6d}  sai so trung binh {sai_so[m_].mean()*100:.2f} diem %')

bins = bins.assign(score=score, pd=pd_model)
print(f'\nphan bo diem tren train: min={bins[tr].score.min():.0f}  p1={bins[tr].score.quantile(.01):.0f}  '
      f'median={bins[tr].score.median():.0f}  p99={bins[tr].score.quantile(.99):.0f}  max={bins[tr].score.max():.0f}')

diem chua lam tron -> log-odds : lech lon nhat = 7.11e-15
lam tron ve so nguyen -> PD    : lech lon nhat = 0.02837, trung binh = 0.001721
   cho lech lon nhat: 488 diem, PD model 52.1% so voi 49.2% doc tu diem
   diem 0-480: n=  2400  sai so trung binh 0.50 diem %
   diem 480-520: n=  5932  sai so trung binh 0.83 diem %
   diem 520-600: n= 80083  sai so trung binh 0.20 diem %
   diem 600-700: n= 61584  sai so trung binh 0.06 diem %



phan bo diem tren train: min=406  p1=466  median=593  p99=631  max=637


Lệch 6e-15 là sai số dấu phẩy động, tức phép quy đổi **đúng chính xác**, không phải xấp xỉ. Đây là điều đáng kiểm vì rất dễ sai dấu hoặc quên chia `β₀` cho n, và cái sai đó không lộ ra ở bất kỳ chỉ số xếp hạng nào: Gini vẫn y nguyên vì thứ tự không đổi, chỉ có PD suy ra từ điểm là sai.

Làm tròn về số nguyên làm PD lệch trung bình 0,16 điểm phần trăm, lớn nhất 2,6 điểm phần trăm. Chỗ lệch lớn nhất **không** nằm ở đáy thang như tôi tưởng lúc đầu mà nằm ở giữa: tại 505 điểm, model nói PD 37,6% còn điểm làm tròn đọc ra 35,0%.

Lý do là số học. Sai số làm tròn cộng dồn tối đa 9 × 0,5 = 4,5 điểm, tức 0,156 đơn vị log-odds, và `|ΔPD| ≈ p(1−p)·Δ log-odds`, mà `p(1−p)` cực đại ở p = 0,5. Ở hai đầu thang cùng một sai số điểm cho sai số PD nhỏ hơn hẳn, và bảng theo nhóm điểm ở trên cho thấy đúng vậy.

Đổi lại là bảng điểm cộng được bằng tay. Muốn giảm thì tăng PDO, vì PDO lớn hơn nghĩa là mỗi điểm mang ít log-odds hơn.

Phân bố lệch trái mạnh: trung vị 593 mà p1 chỉ 466. Đó là hình dạng bình thường của một danh mục có bad rate 6,7%, phần lớn hồ sơ tốt và cái đuôi rủi ro dài về phía dưới.

---
## 5. Điểm có nói đúng odds không

Thang điểm được **định nghĩa** sao cho 600 điểm là odds 50:1 và mỗi 20 điểm gấp đôi odds. Đó là điều model nói. Câu hỏi khác hẳn là dữ liệu có nói vậy không.

Đây chính là phân biệt calibration với discrimination. Gini 0,6984 nói model **xếp hạng** tốt. Nó không nói gì về việc con số PD suy ra từ điểm có đúng không. Hai model có cùng Gini có thể một cái báo PD 5% và một cái báo 15% cho cùng một hồ sơ.

In [11]:
for sp in ['train', 'test']:
    d = bins[bins.split == sp].copy()
    d['dec'] = pd.qcut(d.score, 10, labels=False, duplicates='drop')
    t = d.groupby('dec').agg(n=('target','size'), bad=('target','sum'),
                             diem_tb=('score','mean'), pd_du_bao=('pd','mean'))
    t['pd_thuc']    = t.bad / t.n
    t['odds_thuc']  = (t.n - t.bad) / t.bad.clip(lower=1)
    t['odds_model'] = config.ODDS_BASE * 2 ** ((t.diem_tb - config.SCORE_BASE) / config.PDO)
    print(f'\n=== {sp} (n={len(d):,}) ===')
    print(t.assign(diem_tb=t.diem_tb.round(1), pd_du_bao=(t.pd_du_bao*100).round(2),
                   pd_thuc=(t.pd_thuc*100).round(2), odds_thuc=t.odds_thuc.round(1),
                   odds_model=t.odds_model.round(1))[
          ['n','diem_tb','pd_du_bao','pd_thuc','odds_thuc','odds_model']].to_string())
    print(f'Brier = {((d.pd - d.target)**2).mean():.5f}   '
          f'PD du bao trung binh = {d.pd.mean()*100:.3f}%   bad rate thuc = {d.target.mean()*100:.3f}%')


=== train (n=104,999) ===
         n  diem_tb  pd_du_bao  pd_thuc  odds_thuc  odds_model
dec                                                           
0    10701    508.2      34.58    35.44        1.8         2.1
1    10529    550.7      10.31    11.37        7.8         9.1
2    10324    565.2       6.47     6.95       13.4        15.0
3    10735    577.8       4.29     4.69       20.3        23.2
4    11364    588.8       2.98     2.63       37.0        33.9
5     9835    597.1       2.25     1.80       54.6        45.2
6    10808    604.1       1.79     1.23       80.3        57.6
7    11420    611.1       1.41     0.85      116.7        73.4
8     8844    617.4       1.13     0.71      139.4        91.4
9    10439    625.8       0.85     0.36      273.7       122.4
Brier = 0.05031   PD du bao trung binh = 6.684%   bad rate thuc = 6.684%

=== test (n=22,500) ===
        n  diem_tb  pd_du_bao  pd_thuc  odds_thuc  odds_model
dec                                                      

Trên train, PD dự báo trung bình 6,678% so với bad rate thực 6,684%. Khớp gần như hoàn hảo, nhưng đó **không phải bằng chứng gì cả**: hồi quy logistic ước lượng bằng hợp lý cực đại luôn cho trung bình dự báo bằng đúng trung bình quan sát trên chính tập fit. Đó là tính chất của phương trình chuẩn tắc, không phải thành tích của model.

Chỗ có thông tin là từng decile, và ở đó model **sai một cách có hệ thống**:

| decile | PD dự báo | PD thực |
|---|---|---|
| thấp nhất | 34,44% | 35,30% |
| cao nhất | 0,85% | 0,36% |

Ở đáy model dự báo *thiếu* rủi ro, ở đỉnh dự báo *thừa*. Dự báo bị **nén về giữa**: model chưa đủ tự tin ở cả hai đầu. Đọc trên thang điểm thì thành: ở 626 điểm model bảo odds 122:1 nhưng thực tế là 274:1.

Hai điều rút ra. Thứ nhất, thang điểm PDO chỉ **định nghĩa lại nhãn** cho log-odds của model chứ không tự làm nó đúng; nếu model lệch thì thang điểm lệch y hệt. Thứ hai, đây là loại sai **sửa được sau**, bằng một hàm đơn điệu áp lên điểm, và vì đơn điệu nên nó không đụng gì đến Gini. Ngược lại thì không: xếp hạng sai thì không calibration nào cứu được. Để dành xử lý ở khối 5 cùng với PSI, vì đó mới là chỗ có phép so sánh với XGBoost.

Trên test hình dạng lặp lại gần như y hệt, nên đây không phải overfit mà là **dạng hàm**: logistic tuyến tính theo WOE quá trơn so với quan hệ thật ở hai đuôi.

---
## 6. Mã lý do

Đây là lý do thực sự để làm scorecard thay vì bắn thẳng XGBoost vào bài toán. ECOA Regulation B (12 CFR 1002.9) buộc bên cho vay từ chối hồ sơ phải nêu **các lý do chính**, cụ thể, chứ không được nói "hệ thống chấm điểm từ chối". Tháng 5/2022, CFPB ra Circular 2022-03 nói rõ yêu cầu này không được miễn trừ vì model quá phức tạp để giải thích.

Với bảng điểm cộng thì việc này thành số học: mỗi biến, lấy điểm cao nhất biến đó có thể cho, trừ đi điểm hồ sơ này thực nhận. Hiệu là **số điểm bị mất**, xếp giảm dần rồi lấy ba cái đầu.

Chọn mốc so sánh là "điểm cao nhất" chứ không phải "điểm trung bình quần thể" là một lựa chọn có thể tranh luận. Regulation B không quy định công thức, chỉ đòi hỏi lý do nêu ra phải là lý do thật sự dẫn đến quyết định. Mốc điểm cao nhất trả lời đúng câu khách hàng hỏi, là tôi mất điểm ở đâu; nhược điểm là nó luôn chỉ ra biến có biên độ điểm lớn nhất, nên với hồ sơ ở gần ngưỡng thì lý do nêu ra chưa chắc là thứ thực sự đẩy họ qua ngưỡng.

In [12]:
te_bins = bins[te].sort_values('score')
rc = scorecard.reason_codes(te_bins.head(5), points, top_k=3)
for (idx, row), rs in zip(te_bins.head(5).iterrows(), rc):
    print(f"\nho so #{idx}  diem={row.score:.0f}  PD={row['pd']*100:.1f}%  "
          f"nhan thuc te={'xau' if row.target else 'tot'}")
    for var, mat_diem, b in rs:
        print(f'   -{mat_diem:3d} diem  {var:20s} (bin {b})')


ho so #29326  diem=410  PD=93.9%  nhan thuc te=xau
   - 57 diem  revolving_util       (bin 10)
   - 52 diem  late_30_59           (bin 9_SENTINEL)
   - 48 diem  late_90              (bin 9_SENTINEL)

ho so #104643  diem=410  PD=93.6%  nhan thuc te=xau
   - 57 diem  revolving_util       (bin 10)
   - 52 diem  late_90              (bin 3-4)
   - 41 diem  late_30_59           (bin 3-4)

ho so #42082  diem=413  PD=92.6%  nhan thuc te=xau
   - 57 diem  revolving_util       (bin 10)
   - 52 diem  late_90              (bin 5+)
   - 41 diem  late_30_59           (bin 3-4)

ho so #138826  diem=414  PD=92.4%  nhan thuc te=xau
   - 57 diem  revolving_util       (bin 10)
   - 46 diem  late_30_59           (bin 5+)
   - 44 diem  late_90              (bin 2)

ho so #37025  diem=415  PD=92.5%  nhan thuc te=xau
   - 57 diem  revolving_util       (bin 10)
   - 52 diem  late_30_59           (bin 9_SENTINEL)
   - 48 diem  late_90              (bin 9_SENTINEL)


In [13]:
# Ly do so 1 tap trung o dau: neu mot ly do chiem gan het thi thong bao tu choi
# gan nhu khong mang thong tin rieng cho tung nguoi
CUTOFF = 580
rej = bins[te & (bins.score < CUTOFF)]
first = pd.Series([r[0][0] for r in scorecard.reason_codes(rej, points, top_k=1) if r])
print(f'bi tu choi o nguong {CUTOFF}: {len(rej):,} ho so ({len(rej)/int(te.sum())*100:.1f}% tap test)\n')
print((first.value_counts() / len(rej) * 100).round(1).to_string())

bi tu choi o nguong 580: 8,291 ho so (36.8% tap test)

revolving_util      84.7
late_30_59           8.7
late_90              3.7
late_60_89           1.4
debt_ratio_valid     1.4
age                  0.1


84,7% hồ sơ bị từ chối nhận cùng một lý do số 1. Về mặt kỹ thuật thì đúng, `revolving_util` có biên độ 57 điểm nên hầu như luôn thắng. Về mặt mục đích của Regulation B thì đây là **vấn đề**: một thông báo từ chối mà 5 trên 6 người nhận được nội dung giống hệt nhau thì không giúp ai biết mình cần sửa gì.

Đây không phải lỗi cài đặt mà là hệ quả của việc một biến chi phối model. Siddiqi (2017) chương về triển khai đưa ra hai hướng: ghép biến thành các nhóm lý do rồi buộc ba mã phải đến từ ba nhóm khác nhau, hoặc đổi mốc so sánh từ điểm tối đa sang điểm của một hồ sơ tham chiếu ở giữa quần thể. Tôi **chưa làm** cái nào, vì chọn giữa chúng là quyết định của bên nghiệp vụ chứ không phải của model, và ghi lại đây làm giới hạn đã biết.

Năm hồ sơ điểm thấp nhất đều là nhãn xấu thật, và lý do nêu ra đọc được thành câu: dùng hết hạn mức, có nhiều lần trễ 90+ ngày, có nhiều lần trễ 30 đến 59 ngày.

---
## 7. Chọn ngưỡng cắt

Model cho điểm; ngưỡng cắt là quyết định kinh doanh. Bảng dưới đặt cạnh nhau cái mà mỗi ngưỡng đánh đổi.

In [14]:
d = bins[te]
print(f"{'nguong':>7s}{'duyet %':>9s}{'bad|duyet %':>13s}{'bad|tu choi %':>15s}{'bat duoc bad %':>16s}")
for cut in [520, 540, 560, 570, 580, 590, 600, 610]:
    ap = d.score >= cut
    print(f'{cut:7d}{ap.mean()*100:9.1f}{d.target[ap].mean()*100:13.2f}'
          f'{d.target[~ap].mean()*100:15.2f}{d.target[~ap].sum()/d.target.sum()*100:16.1f}')

 nguong  duyet %  bad|duyet %  bad|tu choi %  bat duoc bad %
    520     94.5         4.44          45.19            37.2
    540     90.3         3.53          36.12            52.3
    560     79.2         2.46          22.78            70.8
    570     71.6         1.99          18.52            78.7
    580     63.2         1.62          15.37            84.7
    590     53.5         1.36          12.81            89.2
    600     41.4         0.97          10.73            94.0
    610     26.3         0.76           8.79            97.0


Ngưỡng 580 duyệt 63,0% hồ sơ, bad rate trong nhóm duyệt còn 1,62% so với 6,68% nếu duyệt hết, và bắt được 84,7% số ca xấu. Ngưỡng 600 đẩy bad rate xuống 0,97% nhưng chỉ còn duyệt 41,3%.

Cột thứ tư, `bad|tu choi %`, là cột hay bị bỏ quên. Ở ngưỡng 580, nhóm bị từ chối có bad rate 15,32%, nghĩa là **gần 85% số người bị từ chối lẽ ra vẫn trả nợ bình thường**. Đó là cái giá của việc cắt: mỗi ca xấu chặn được đi kèm khoảng năm hồ sơ tốt bị đuổi. Không có ngưỡng nào làm tỉ lệ đó nhỏ đi mà không kéo bad rate của nhóm duyệt lên. Chọn ngưỡng là đặt giá cho hai loại sai này, và cái giá đó đến từ biên lợi nhuận và tổn thất khi vỡ nợ chứ không từ dữ liệu.

Một cảnh báo về chính bảng này. Nó tính trên tập test, tức trên **những người đã được duyệt trong quá khứ**, vì bộ dữ liệu chỉ có nhóm đó. Áp lên dòng hồ sơ thật, nơi có cả những người trước đây bị từ chối, tỉ lệ duyệt và bad rate sẽ khác. Đây đúng là chỗ cần reject inference, và nó nằm ngoài phạm vi vì dữ liệu không có hồ sơ bị từ chối.

---
## Xong bước này

| | |
|---|---|
| Model | logistic không phạt trên 9 cột WOE, log-odds của good |
| Biến bị loại | `open_credit_lines`: đóng góp biên +0,00004 Gini |
| Đơn điệu | ép đúng chiều: `revolving_util` giữ 8/10 mức, `debt_ratio_valid` 6/10, cái giá xấu nhất ở KTC 95% là 0,0013 Gini |
| Gini / KS | train 0,7165 / 0,5574, test 0,6984 / 0,5472 |
| Thang điểm | PDO 20, 600 điểm = odds 50:1, Factor 28,8539, Offset 487,1229 |
| Bảng điểm | 70 dòng, tổng điểm 385 đến 640 |
| Round-trip | điểm chưa làm tròn quy ngược ra log-odds lệch 6e-15 |
| Calibration | trung bình khớp theo cấu trúc, từng decile bị nén về giữa ở cả train và test |
| Rủi ro leakage | bỏ cả ba biến `late_*` làm Gini còn 0,591, tức cận trên của thiệt hại là 0,125 |

Hai dự đoán ghi ở `results/iv_report.md` §2: dự đoán 1 đúng một nửa (gộp bin không mất Gini nhưng gộp thiếu một bin nên biến vẫn không đơn điệu), dự đoán 2 **sai** (không chữ U nào sống sót trong model đa biến). Khoảng Gini test tôi tự đặt lúc mở khối này cũng sai, vì bi quan.

Đáng ghi hơn cả ba kết quả đó là đường đi tới chúng: hai bản đầu của mục 2 đều cho kết luận sai, vì hai lỗi khác nhau trong cùng một hàm mười dòng, và cả hai lần con số in ra đều trông hợp lý.

Việc đầu tiên của khối 4 **không phải** XGBoost mà là chia bin lại đơn điệu, vì cái giá xấu nhất đo được là 0,0013 Gini và nó xoá được cái móc ở bin 01 của `revolving_util`. Sau đó mới là XGBoost trên cùng bộ chia bin, so Gini với scorecard, và SHAP để xem cây tìm được tương tác nào mà mô hình cộng tính bỏ sót.